# 07 - Error analysis: where the errors are and why

Self-contained notebook that only reads results of the earlier notebooks (`data/corners.csv`, `data/gate_ml.csv`, `data/gate_devices.csv`, `data/circuit_ml.csv`) and no simulator. It answers the question of the project text: *where do the largest errors occur (which gate, input combination or corner conditions, for example high temperature or low vth0) and why?* Errors are relative errors `(ML - SPICE) / SPICE` of the leakage; "median" and "95th percentile" are of their absolute value, "bias" is the mean signed error.

## 1. Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR, RESULT_DIR = ROOT / "data", ROOT / "results"
KEY = ["gate", "pattern", "corner"]

corners = pd.read_csv(DATA_DIR / "corners.csv")
gate = pd.read_csv(DATA_DIR / "gate_ml.csv", dtype={"pattern": str}).merge(corners.drop(columns=["vdd", "temp"]), on="corner")
dev = pd.read_csv(DATA_DIR / "gate_devices.csv", dtype={"pattern": str})
circ = pd.read_csv(DATA_DIR / "circuit_ml.csv").merge(corners.drop(columns=["vdd", "temp"]), on="corner")
gate["p_vth0_mag"], circ["p_vth0_mag"] = gate["p_vth0"].abs(), circ["p_vth0"].abs()
print("gate-level points:", len(gate), "| transistor instances:", len(dev), "| circuit points:", len(circ))

## 2. Gate level: which gates and input patterns
The gate results table of notebook 05, ranked by the median error (all 200 corners per case). The bias shows the direction of the error.

In [ ]:
def stats(d, err="err"):
    a = d[err].abs()
    return pd.Series({"n": len(d), "bias [%]": d[err].mean() * 100, "median [%]": a.median() * 100, "95th pct [%]": a.quantile(0.95) * 100, "mean [%]": a.mean() * 100})


by_case = gate.groupby(["gate", "pattern"], sort=False).apply(stats, include_groups=False).sort_values("median [%]", ascending=False)
by_gate = gate.groupby("gate", sort=False).apply(stats, include_groups=False)
print("by gate type:")
display(by_gate.round(2))
print("by gate and input pattern, largest median error first:")
by_case.round(2)

## 3. Gate level: which corner conditions
Median and 95th percentile of the absolute error, and the bias, in bins of temperature, VDD and the process parameters of the corner (all 18 cases pooled; the bins of VDD and of the process parameters are terciles of the 200 corners; for the PMOS `vth0` the magnitude is used, so "low" means a low threshold).

In [ ]:
FACTORS = [("temp", "temperature [degC]", [-40, -12.5, 15, 42.5, 70, 97.5, 125]), ("vdd", "VDD [V]", 3), ("n_vth0", "NMOS vth0 [V]", 3), ("p_vth0_mag", "PMOS |vth0| [V]", 3),
           ("n_toxe", "NMOS toxe [m]", 3), ("p_toxe", "PMOS toxe [m]", 3)]


def binned(df, col, bins, err="err"):
    cut = pd.cut(df[col], bins, include_lowest=True) if isinstance(bins, list) else pd.qcut(df[col], bins)
    return df.groupby(cut, observed=True).apply(stats, err=err, include_groups=False)


tabs = {name: binned(gate, col, bins) for col, name, bins in FACTORS}
for name, t in tabs.items():
    t.index = [f"{i.left:.3g} .. {i.right:.3g}" for i in t.index]
    print(name)
    display(t.round(2))

In [ ]:
COL = {"median": "#2a78d6", "p95": "#eb6834"}
fig, ax = plt.subplots(2, 3, figsize=(14, 6.5))
for a, (name, t) in zip(ax.ravel(), tabs.items()):
    x = np.arange(len(t))
    a.plot(x, t["median [%]"], "o-", color=COL["median"], ms=7, mec="white", mew=1.2, label="median |error|")
    a.plot(x, t["95th pct [%]"], "s-", color=COL["p95"], ms=7, mec="white", mew=1.2, label="95th percentile")
    a.plot(x, t["bias [%]"], "^--", color="0.4", ms=6, mec="white", mew=1.0, lw=1, label="bias")
    a.axhline(0, color="0.8", lw=0.8)
    a.set_xticks(x); a.set_xticklabels([s.replace(" .. ", "\n") for s in t.index], fontsize=7.5)
    a.set_xlabel(name); a.set_ylabel("error [%]")
    a.grid(axis="y", color="0.92", lw=0.8); a.set_axisbelow(True); a.spines[["top", "right"]].set_visible(False)
handles, labels = ax[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(RESULT_DIR / "gate_error_factors.png", dpi=110)
plt.show()

## 4. Gate level: which transistor model and which leakage component
The leakage of a gate is the sum of the terms of its transistors (channel, gate, body; notebook 05). With the same voltages, the transistor currents from the ML models and from SPICE give two values of every term, so the error of the gate leakage can be traced back to a model and a component: for each combination, `sum |ML - SPICE| / sum(gate leakage)` over all 3,600 operating points (the **absolute error budget**, in %), and the same with signs (the **bias**; the sum of the signed terms is the total bias of the table above).

In [ ]:
dv = dev.merge(gate[KEY + ["leak_total"]], on=KEY)
total = gate["leak_total"].sum()
rows = []
for case, g in dv.groupby("case"):
    for comp in ("channel", "gate", "body"):
        d = g[f"ml_{comp}"] - g[f"spice_{comp}"]
        rows.append(dict(model=case, component=comp, abs_budget=d.abs().sum() / total * 100, bias=d.sum() / total * 100, n=len(g)))
budget = pd.DataFrame(rows)
display(budget.pivot(index="model", columns="component", values="abs_budget").round(2).assign(total=lambda t: t.sum(axis=1)).rename_axis("absolute error budget [% of the summed leakage]"))
display(budget.pivot(index="model", columns="component", values="bias").round(2).assign(total=lambda t: t.sum(axis=1)).rename_axis("bias [% of the summed leakage]"))
print("check: summed bias of all terms =", round(budget["bias"].sum(), 3), "% | bias of the sum of the gate leakages =", round((gate["ml_leak"].sum() - total) / total * 100, 3), "%")

**Share of the SPICE leakage by term** (how much each model / component contributes to the leakage itself, to read the budget against):

In [ ]:
share = pd.DataFrame([dict(model=case, component=comp, share=g[f"spice_{comp}"].sum() / total * 100) for case, g in dv.groupby("case") for comp in ("channel", "gate", "body")])
share.pivot(index="model", columns="component", values="share").round(1).assign(total=lambda t: t.sum(axis=1))

## 5. The ten worst gate operating points

In [ ]:
worst = gate.reindex(gate["err"].abs().sort_values(ascending=False).index).head(10)[KEY + ["temp", "vdd", "n_vth0", "p_vth0", "leak_total", "ml_leak", "err"]].copy()
dom = []
for _, r in worst.iterrows():
    g = dv[(dv.gate == r.gate) & (dv.pattern == r.pattern) & (dv.corner == r.corner)]
    best = max(((abs(row[f"ml_{c}"] - row[f"spice_{c}"]), f"{row['device']} ({row['case']}) {c}") for _, row in g.iterrows() for c in ("channel", "gate", "body")))
    dom.append(best[1])
worst["largest term error"] = dom
worst.assign(err=lambda t: (t["err"] * 100).round(1), leak_total=lambda t: t["leak_total"].map("{:.2e}".format), ml_leak=lambda t: t["ml_leak"].map("{:.2e}".format))

## 6. Test circuit
The error of the hierarchical prediction is the sum of two parts: the **composition floor** (the sum of the isolated-gate SPICE leakages against the full-circuit SPICE leakage, i.e. what perfect gate models would give: loading and non-ideal gate inputs) and the **gate-model error** (ML gate leakage against the isolated-gate SPICE sum). Both by corner condition:

In [ ]:
rows = []
for col, name, bins in FACTORS[:4]:
    cut = pd.cut(circ[col], bins, include_lowest=True) if isinstance(bins, list) else pd.qcut(circ[col], bins)
    for iv, g in circ.groupby(cut, observed=True):
        rows.append({"factor": name, "bin": f"{iv.left:.3g} .. {iv.right:.3g}",
                     "ML: median |err| [%]": g["err"].abs().median() * 100, "ML: 95th pct [%]": g["err"].abs().quantile(0.95) * 100, "ML: bias [%]": g["err"].mean() * 100,
                     "floor: median |err| [%]": g["err_floor"].abs().median() * 100, "floor: bias [%]": g["err_floor"].mean() * 100,
                     "gate models: bias [%]": g["err_gates"].mean() * 100, "gate models: median |err| [%]": g["err_gates"].abs().median() * 100})
circ_tab = pd.DataFrame(rows).set_index(["factor", "bin"])
circ_tab.to_csv(RESULT_DIR / "circuit_error_by_corner.csv")
circ_tab.round(2)

**By input vector** (200 corners each): the vectors with the largest and smallest median error of the hierarchical prediction.

In [ ]:
bv = circ.groupby("vector").apply(lambda g: pd.Series({"a": int(g.name) & 7, "b": (int(g.name) >> 3) & 7, "cin": int(g.name) >> 6, "median |err| [%]": g["err"].abs().median() * 100,
                                                       "bias [%]": g["err"].mean() * 100, "SPICE median [A]": g["leak_total"].median()}), include_groups=False)
print("largest median error:")
display(bv.sort_values("median |err| [%]", ascending=False).head(6).round(3))
print("smallest median error:")
bv.sort_values("median |err| [%]").head(6).round(3)